In [7]:
import os
from typing import List
from pydantic import BaseModel
from langchain_openai import OpenAIEmbeddings
from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import TextLoader,WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph,END
from langchain_core.documents import Document

In [2]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
llm=init_chat_model("openai:gpt-4o")

In [8]:
docs=TextLoader("AI.txt",encoding="utf-8").load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=50)
chunks=text_splitter.split_documents(docs)
embedding=OpenAIEmbeddings()
vectorstore=FAISS.from_documents(chunks,embedding)
retriever=vectorstore.as_retriever()

In [12]:
### Define Agent State
class IterativeRAGState(BaseModel):
    question:str
    refined_question:str=''
    retrieved_docsList:List[Document]=[]
    answer:str=''
    verified:bool=False
    attempts:int=0

In [13]:
### Retrive docs
def retrieve_docs(state: IterativeRAGState):
    query=state.refined_question
    docs=retriever.invoke(query)
    return state.model_copy(update={"retrieved_docsList":docs})

In [32]:
### Rflect and verify
def generate_answer(state: IterativeRAGState):
    context="\\n".join(doc.page_content for doc in state.retrieved_docsList)
    prompt=f"""Based on the following context,answer the question:
    context: {context}
    Question: {state.question}
    """
    response=llm.invoke(prompt)
    return state.model_copy(update={"answer":response.content,"attempts":state.attempts+1})


In [33]:
### Reflect on answer
def reflect_on_answer(state: IterativeRAGState):
    prompt=f"""Based on the following context,reflect on the answer:
    Question:{state.question}
    Answer:{state.answer}
    Response 'YES' if the answer is correct, otherwise 'NO
    """
    feedback=llm.invoke(prompt).content.lower()
    verified="Yes" in feedback
    return state.model_copy(update={"verified":verified})

In [34]:
### Rfine Query
def refine_query(state: IterativeRAGState):
    prompt=f"""Based on the following context,refine the query:
    original_question:{state.question}
    current_answer:{state.answer}
    """
    refined_query=llm.invoke(prompt).content.strip()
    return state.model_copy(update={"question":refined_query})

In [35]:
### state Graph
builder=StateGraph(IterativeRAGState)
builder.add_node("retrieve",retrieve_docs)
builder.add_node("answer",generate_answer)
builder.add_node("reflect",reflect_on_answer)
builder.add_node("refine",refine_query)

builder.set_entry_point("retrieve")
builder.add_edge("retrieve","answer")
builder.add_edge("answer","reflect")

builder.add_conditional_edges(
    "reflect",
    lambda s:END if s.verified or s.attempts>2 else "refine"
)
builder.add_edge("refine","retrieve")
builder.add_edge("answer",END)
graph=builder.compile()

In [36]:
query="what are transformers"
initial_state=IterativeRAGState(question=query)
final_state=graph.invoke(initial_state)
print("final answer:",final_state["answer"])
print("verified:",final_state["verified"])
print("attempts:",final_state["attempts"])

final answer: Transformer models have revolutionized the field of machine learning, particularly in natural language processing (NLP). Let's delve into their role and impact:

1. **Introduction to Transformer Models:**
   - Transformer models, introduced in the paper "Attention is All You Need" by Vaswani et al. in 2017, are a type of neural network architecture primarily designed for handling sequential data, such as text.
   - The key innovation in transformers is the self-attention mechanism, which allows the model to weigh the importance of different words in a sentence relative to one another, regardless of their position. This enables the model to capture long-range dependencies more effectively than previous models like RNNs and LSTMs.

2. **Applications in NLP:**
   - **Language Understanding:** Transformers are used in tasks like sentiment analysis, named entity recognition, and part-of-speech tagging.
   - **Machine Translation:** These models have significantly improved mach

In [38]:
final_state

{'question': 'Refined query: Could you elaborate on the role and impact of transformer models in machine learning, with a focus on their application in natural language processing (NLP)?',
 'refined_question': '',
 'retrieved_docsList': [Document(id='3a8d6414-ea4d-4fd2-98d3-928a4a63596b', metadata={'source': 'AI.txt'}, page_content="AI myth versus reality\nLet's clear up some common misconceptions about AI."),
  Document(id='bbf0ba57-1581-4b6c-98a1-af3bb9917d39', metadata={'source': 'AI.txt'}, page_content='Want to see more use cases for AI? Find 1,000+ real-world gen AI use cases over at the Google Cloud Blog.'),
  Document(id='aea09abd-d408-4bb0-bbb8-42c713304592', metadata={'source': 'AI.txt'}, page_content="Want to learn how to get started with AI? Take the free beginner's introduction to generative AI."),
  Document(id='84c841f6-1c40-4a49-a089-868d50469bf8', metadata={'source': 'AI.txt'}, page_content="In your daily life: Your smartphone's virtual assistant, personalized recommend